In [52]:
import polars as pl
import polars.selectors as cs
import os
import pyarrow.dataset as pads
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from joblib import Parallel, delayed

In [ ]:
def readDataSubset(table_type, data_path = ""):
    if table_type not in ["ball-positions", "ball-events", "player-positions", "lineups", "game-info"]:
        print("Invalid")
        return -1 

    if table_type == 'lineups' or table_type == "game-info":
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + 'csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [54]:
selected_game_str = "y1_d061_VKA_PHD"

# Data Frame for ball events filtered down to game str
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Data Frame for ball position filtered down to game str
ball_pos_subset = readDataSubset('ball-positions')
ball_pos = ball_pos_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
ball_pos_df = pl.from_pandas(ball_pos)

# Data Frame for player position filtered down to game str
player_pos_subset = readDataSubset("player-positions")
player_pos = player_pos_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
player_pos_df = pl.from_pandas(player_pos)

# Should I Steal Model

### Model to predict the best oppurtunity to steal
- based on these features
    - Where is the ball located
    - What base is runner on
    - How many outs in innning
    - Defensive player positions

- Build with runner on 1st base only with feature being Defensive player positions
    - First : Find all plays where the is a runner on 1st base (player id == 11)
    - Second : Find where all successful steals
    - Third : analysis why those where successful steals based on defensive positions
    - Fourth : build model on this data and use to predict new data

In [55]:
# Find all plays where someone is on 1st base

on_1st_base_df = player_pos_df.filter(pl.col("player_id") == 11)

# Find all player where someone is on 2nd base

on_2nd_base_df = player_pos_df.filter(pl.col("player_id") == 12)

# Find all plays where someone is on 3rd base

on_3rd_base_df = player_pos_df.filter(pl.col("player_id") == 13)

# Make it where there is only someone on 1st; no players on second or third

# Make a list that combines the game_str and play_id
on_1st_base = on_1st_base_df["game_string"] + "_" + on_1st_base_df["play_per_game"]
on_2nd_base = on_2nd_base_df["game_string"] + "_" + on_2nd_base_df["play_per_game"]
on_3rd_base = on_3rd_base_df["game_string"] + "_" + on_3rd_base_df["play_per_game"]

In [56]:
on_1st_base

game_string
str
"""y1_d061_VKA_PHD_35"""
"""y1_d061_VKA_PHD_35"""
"""y1_d061_VKA_PHD_35"""
"""y1_d061_VKA_PHD_35"""
"""y1_d061_VKA_PHD_35"""
…
"""y1_d061_VKA_PHD_224"""
"""y1_d061_VKA_PHD_224"""
"""y1_d061_VKA_PHD_224"""


In [57]:
player_pos_w_game_str_ppg = player_pos_df.with_columns(pl.concat_str([pl.col("game_string"), pl.col("play_per_game")],
                                                                      separator = "_").alias("game_str_ppg"))

player_pos_bases = player_pos_w_game_str_ppg.filter(
    # Filter to only game_str_ppg in on_1st, 2nd, 3rd
    # Convert from one column data frame to list
    pl.col("game_str_ppg").is_in(on_1st_base.unique().to_list()),
    pl.col("game_str_ppg").is_in(on_2nd_base.unique().to_list()),
    pl.col("game_str_ppg").is_in(on_3rd_base.unique().to_list()),
)

player_on_bases = player_pos_bases.sort(
    # order chronologically
    ["game_string", "play_per_game", "timestamp"]
).group_by(
    # group by player id
    ["player_id"]
).agg(
    # get the rows that only have bases occupied
    (pl.col("player_id") == 11).any().alias("on_first"),
    (pl.col("player_id") == 12).any().alias("on_second"),
    (pl.col("player_id") == 13).any().alias("on_third")
)

on_based_df = player_pos_w_game_str_ppg.join(player_on_bases,
                                    on = ["player_id"], how = "left")

In [58]:
new_df = on_based_df.drop_nulls()
new_df

game_string,play_per_game,timestamp,player_id,field_x,field_y,home_team,away_team,year,day,game_str_ppg,on_first,on_second,on_third
str,i64,i64,i64,f64,f64,str,str,str,str,str,bool,bool,bool
"""y1_d061_VKA_PHD""",1,63614,1,-0.3522,56.6067,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_1""",false,false,false
"""y1_d061_VKA_PHD""",1,63614,10,2.0787,-0.3228,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_1""",false,false,false
"""y1_d061_VKA_PHD""",1,63614,15,86.8296,84.738,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_1""",false,false,false
"""y1_d061_VKA_PHD""",1,63614,18,78.3852,53.0442,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_1""",false,false,false
"""y1_d061_VKA_PHD""",1,63614,19,-85.563,64.1982,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_1""",false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""y1_d061_VKA_PHD""",232,7799114,5,-54.3486,79.635,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_232""",false,false,false
"""y1_d061_VKA_PHD""",232,7799114,6,-30.9087,156.3972,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_232""",false,false,false
"""y1_d061_VKA_PHD""",232,7799114,7,-94.5504,321.3765,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_232""",false,false,false


### **Found where all pays == 11 in new_df**

In [59]:
#test = new_df.filter(pl.col("player_id") == 11)
#test = new_df.filter(pl.col("player_id") == 12)
#test = new_df.filter(pl.col("player_id") == 13)
test = new_df.filter(pl.col("on_first") | pl.col("on_second") | pl.col("on_third"))
#test = new_df.filter(pl.col("on_first" & "on_second"))
test

game_string,play_per_game,timestamp,player_id,field_x,field_y,home_team,away_team,year,day,game_str_ppg,on_first,on_second,on_third
str,i64,i64,i64,f64,f64,str,str,str,str,str,bool,bool,bool
"""y1_d061_VKA_PHD""",12,438264,12,-12.3498,122.2041,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_12""",false,true,false
"""y1_d061_VKA_PHD""",12,438314,12,-12.3984,122.1342,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_12""",false,true,false
"""y1_d061_VKA_PHD""",12,438364,12,-12.447,122.064,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_12""",false,true,false
"""y1_d061_VKA_PHD""",12,438414,12,-12.4953,121.9941,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_12""",false,true,false
"""y1_d061_VKA_PHD""",12,438464,12,-12.5436,121.9239,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_12""",false,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""y1_d061_VKA_PHD""",232,7798914,12,-38.7036,33.9219,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_232""",false,true,false
"""y1_d061_VKA_PHD""",232,7798964,12,-38.1264,33.3273,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_232""",false,true,false
"""y1_d061_VKA_PHD""",232,7799014,12,-37.5495,32.7333,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_232""",false,true,false


### **Find all plays where some one appears on second after being on first** (maybe check where someone moves to second between 1 - 5 plays?)